In [2]:
pip install yfinance

  Preparing metadata (setup.py): started
  Preparing metadata (setup.py): finished with status 'done'
   ---------------------------------------- 0.0/1.6 MB ? eta -:--:--
   ---------------------------------------- 1.6/1.6 MB 10.8 MB/s eta 0:00:00
  Created wheel for multitasking: filename=multitasking-0.0.12-py3-none-any.whl size=15703 sha256=b3f7ca7b6140ff9c4e120ab27a6375bf9ca4429218f49211159938b006fc6e7a
  Stored in directory: c:\users\lucas\appdata\local\pip\cache\wheels\42\d6\84\bf57a755f4569494cd00de4bb46ef064874823f4d19c82e960
Successfully built multitasking

   ----------- ---------------------------- 2/7 [websockets]
   ----------------- ---------------------- 3/7 [protobuf]
   ---------------------- ----------------- 4/7 [frozendict]
   ---------------------------------- ----- 6/7 [yfinance]
   ---------------------------------------- 7/7 [yfinance]

Note: you may need to restart the kernel to use updated packages.


  DEPRECATION: Building 'multitasking' using the legacy setup.py bdist_wheel mechanism, which will be removed in a future version. pip 25.3 will enforce this behaviour change. A possible replacement is to use the standardized build interface by setting the `--use-pep517` option, (possibly combined with `--no-build-isolation`), or adding a `pyproject.toml` file to the source tree of 'multitasking'. Discussion can be found at https://github.com/pypa/pip/issues/6334


In [ ]:
import yfinance as yf
import pandas as pd
import os

# carpeta destino
path = r"C:\Users\lucas\Downloads\GdP\Implementacion\Bases de datos"

# universos
byma = ["GGAL.BA","YPFD.BA","PAMP.BA","TXAR.BA","ALUA.BA","CEPU.BA","LOMA.BA",
        "BMA.BA","BBAR.BA","SUPV.BA","COME.BA","EDN.BA","TGNO4.BA","TGSU2.BA","VALO.BA"]

cedears = ["AAPL","MSFT","AMZN","GOOGL","META","NVDA","TSLA","BRK-B","JPM",
           "V","MA","KO","PEP","XOM","UNH"]

etfs = ["SPY","QQQ","IWM","EEM","EFA","GLD","SLV","TLT","IEF","HYG","VNQ"]

tickers = byma + cedears + etfs

# descarga completa
prices = yf.download(tickers, start="2015-01-01", auto_adjust=True)["Close"]


# metadata de activos
meta = pd.DataFrame({
    "ticker": tickers,
    "asset_type": ["BYMA"]*len(byma) + ["CEDEAR"]*len(cedears) + ["ETF"]*len(etfs)
})

# guardar archivos
prices.to_csv(os.path.join(path, "prices_all_assets.csv"))
meta.to_csv(os.path.join(path, "assets_metadata.csv"), index=False)

print("✔ Base completa guardada")


[*********************100%***********************]  41 of 41 completed
C:\Users\lucas\AppData\Local\Temp\ipykernel_27800\4133359937.py:23: FutureWarning: The default fill_method='pad' in DataFrame.pct_change is deprecated and will be removed in a future version. Either fill in any non-leading NA values prior to calling pct_change or specify 'fill_method=None' to not fill NA values.
  returns = prices.pct_change().dropna()


✔ Base completa guardada


In [4]:
print("Fecha inicio dataset:", prices.dropna().index.min())
print("Fecha fin dataset:", prices.index.max())


Fecha inicio dataset: 2017-11-01 00:00:00
Fecha fin dataset: 2026-02-13 00:00:00


In [5]:
prices_clean = prices.dropna()

print("\nResumen dataset final")
print("---------------------")
print("Fecha inicio:", prices_clean.index.min().date())
print("Fecha fin:", prices_clean.index.max().date())
print("Cantidad días:", len(prices_clean))
print("Cantidad activos:", prices_clean.shape[1])



Resumen dataset final
---------------------
Fecha inicio: 2017-11-01
Fecha fin: 2026-02-13
Cantidad días: 1964
Cantidad activos: 41


In [ ]:
import pandas as pd
import os

path = r"C:\Users\lucas\Downloads\GdP\Implementacion\Bases de datos"

prices = pd.read_csv(os.path.join(path,"prices_all_assets.csv"), index_col=0, parse_dates=True)

prices_2018_2025 = prices.loc["2018-01-01":"2025-12-31"]
returns_2018_2025 = returns.loc["2018-01-01":"2025-12-31"]

prices_2018_2025.to_csv(os.path.join(path,"prices_2018_2025.csv"))

print("✔ Ventana 2018–2025 guardada")


✔ Ventana 2018–2025 guardada


In [ ]:
import pandas as pd
import numpy as np
import os

path = r"C:\Users\lucas\Downloads\GdP\Implementacion\Bases de datos"

# ===============================
# 1) PRECIOS EN ARS
# ===============================
prices_ars = pd.read_csv(os.path.join(path,"prices_2018_2025.csv"),
                         index_col=0, parse_dates=True)

prices_ars = prices_ars.ffill()

# ===============================
# 2) DOLAR CCL
# ===============================
ccl = pd.read_csv(os.path.join(path,"DOLAR CCL - Cotizaciones historicas.csv"))

# adaptá nombres si cambian
ccl.columns = [c.lower() for c in ccl.columns]
ccl.rename(columns={"fecha":"date","cierre":"ccl"}, inplace=True)

ccl["date"] = pd.to_datetime(ccl["date"])
ccl = ccl.set_index("date")

# alinear calendario con precios
ccl = ccl.reindex(prices_ars.index).ffill()

# ===============================
# 3) CONVERTIR TODO A USD
# ===============================
prices_usd = prices_ars.divide(ccl["ccl"], axis=0)

# ===============================
# 4) LOG RETURNS EN USD
# ===============================
returns_usd = np.log(prices_usd / prices_usd.shift(1)).dropna()

# ===============================
# 5) GUARDAR DATASET FINAL
# ===============================
prices_usd.to_csv(os.path.join(path,"prices_USD.csv"))
returns_usd.to_csv(os.path.join(path,"returns_USD.csv"))

print("✔ DATASET FINAL EN USD")
print("Rango:", prices_usd.index.min().date(), "→", prices_usd.index.max().date())
print("Activos:", prices_usd.shape[1])

✔ DATASET FINAL EN USD
Rango: 2018-01-01 → 2025-12-31
Activos: 41


In [10]:
import pandas as pd
import os

path = r"C:\Users\lucas\Downloads\GdP\Implementacion\Bases de datos"

byma = ["GGAL.BA","YPFD.BA","PAMP.BA","TXAR.BA","ALUA.BA","CEPU.BA","LOMA.BA",
        "BMA.BA","BBAR.BA","SUPV.BA","COME.BA","EDN.BA","TGNO4.BA","TGSU2.BA","VALO.BA"]

cedears = ["AAPL","MSFT","AMZN","GOOGL","META","NVDA","TSLA","BRK-B","JPM",
           "V","MA","KO","PEP","XOM","UNH"]

etfs = ["SPY","QQQ","IWM","EEM","EFA","GLD","SLV","TLT","IEF","HYG","VNQ"]

meta = pd.DataFrame({
    "ticker": byma + cedears + etfs,
    "asset_class": ["Argentina Equity"]*len(byma) +
                   ["US Equity"]*len(cedears) +
                   ["ETF"]*len(etfs)
})

meta.to_csv(os.path.join(path,"assets_metadata.csv"), index=False)

print("✔ assets_metadata.csv creado")


✔ assets_metadata.csv creado


In [12]:
import pandas as pd
import numpy as np
import os

path = r"C:\Users\lucas\Downloads\GdP\Implementacion\Bases de datos"

# 1) Precios mixtos (ARS para .BA, USD para US/ETF)
prices = pd.read_csv(os.path.join(path, "prices_2018_2025.csv"),
                     index_col=0, parse_dates=True).sort_index()
prices = prices.ffill()

# 2) CCL
ccl = pd.read_csv(os.path.join(path, "DOLAR CCL - Cotizaciones historicas.csv"))
ccl.columns = [c.lower() for c in ccl.columns]
ccl.rename(columns={"fecha": "date", "cierre": "ccl"}, inplace=True)
ccl["date"] = pd.to_datetime(ccl["date"])
ccl = ccl.set_index("date").sort_index()

# Alinear a calendario de precios
ccl = ccl.reindex(prices.index).ffill()

# 3) Convertir SOLO activos argentinos (.BA) a USD
prices_usd = prices.copy()
arg_assets = [col for col in prices.columns if col.endswith(".BA")]

prices_usd[arg_assets] = prices_usd[arg_assets].divide(ccl["ccl"], axis=0)

# 4) Log returns
returns_usd = np.log(prices_usd / prices_usd.shift(1)).dropna()

# 5) Guardar
prices_usd.to_csv(os.path.join(path, "prices.csv"))
returns_usd.to_csv(os.path.join(path, "returns.csv"))

print("✔ Dataset USD corregido")
print("Rango:", prices_usd.index.min().date(), "→", prices_usd.index.max().date())
print("Activos:", prices_usd.shape[1])
print("Convertidos a USD (ARS→USD):", len(arg_assets))
print("Dejados en USD (US/ETF):", prices_usd.shape[1] - len(arg_assets))



✔ Dataset USD corregido
Rango: 2018-01-01 → 2025-12-31
Activos: 41
Convertidos a USD (ARS→USD): 15
Dejados en USD (US/ETF): 26


In [14]:
pip install scipy


Note: you may need to restart the kernel to use updated packages.


In [ ]:

python src/main.p

SyntaxError: invalid syntax (708034916.py, line 1)